# Experiment 01: Baseline U-Net

**Date:** 2025-10-08  
**Author:** CAP5410 Project Team  
**Objective:** Establish baseline performance for optic disc and cup segmentation using standard U-Net architecture

---

## 📋 Experiment Overview

### Hypothesis
A standard U-Net architecture with 64 base features can effectively segment optic disc and optic cup from fundus images.

### Configuration
- **Model:** U-Net (64 base features)
- **Preprocessing:** None (baseline)
- **Dataset:** REFUGE (cropped masks)
- **Training samples:** 400
- **Validation samples:** 400
- **Test samples:** 400

### Hyperparameters
```python
{
    'epochs': 50,
    'batch_size': 8,
    'learning_rate': 1e-4,
    'optimizer': 'Adam',
    'weight_decay': 1e-4,
    'loss': 'BCEWithLogitsLoss',
}
```

---

## 1️⃣ Environment Setup

In [ ]:
import sys
import os
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / 'src'))

print(f"Project root: {project_root}")
print(f"Working directory: {Path.cwd()}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

# Project imports
from models.unet import UNet
from data_loader.dataset import RetinaDataset, RetinaDatasetTest
from experiments.utils import (
    train_model,
    test_model,
    plot_training_curves,
    visualize_predictions,
    visualize_data_samples,
    save_config,
    save_history,
    save_test_results,
    print_test_results,
    print_model_info,
)

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2️⃣ Configuration

In [ ]:
# Experiment configuration
CONFIG = {
    # Model
    'model': 'unet',
    'base_features': 64,
    'bilinear': False,
    
    # Data
    'data_dir': '../../datasets/REFUGE',
    'train_csv': '../../datasets/REFUGE/REFUGETrain.csv',
    'val_csv': '../../datasets/REFUGE/REFUGE1Val.csv',
    'test_csv': '../../datasets/REFUGE/REFUGE1Test.csv',
    'use_cropped': True,
    'use_clahe': False,  # Baseline: no CLAHE
    
    # Training
    'epochs': 50,
    'batch_size': 8,
    'learning_rate': 1e-4,
    'weight_decay': 1e-4,
    'num_workers': 4,
    
    # Output
    'output_dir': './results',
}

# Create output directory
os.makedirs(CONFIG['output_dir'], exist_ok=True)
os.makedirs(f"{CONFIG['output_dir']}/visualizations", exist_ok=True)

# Save configuration
save_config(CONFIG, f"{CONFIG['output_dir']}/config.json")

## 3️⃣ Data Loading

In [ ]:
# Create datasets
print("Loading datasets...")

train_dataset = RetinaDataset(
    csv_file=CONFIG['train_csv'],
    root_dir=CONFIG['data_dir'],
    use_cropped=CONFIG['use_cropped'],
    use_clahe=CONFIG['use_clahe'],
)

val_dataset = RetinaDataset(
    csv_file=CONFIG['val_csv'],
    root_dir=CONFIG['data_dir'],
    use_cropped=CONFIG['use_cropped'],
    use_clahe=CONFIG['use_clahe'],
)

test_dataset = RetinaDatasetTest(
    csv_file=CONFIG['test_csv'],
    root_dir=CONFIG['data_dir'],
    use_cropped=CONFIG['use_cropped'],
    use_clahe=CONFIG['use_clahe'],
)

print(f"✅ Train samples: {len(train_dataset)}")
print(f"✅ Validation samples: {len(val_dataset)}")
print(f"✅ Test samples: {len(test_dataset)}")

In [ ]:
# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=CONFIG['num_workers'],
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=CONFIG['num_workers'],
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=CONFIG['num_workers'],
    pin_memory=True
)

print("✅ Data loaders created")

### 📊 Data Visualization

In [ ]:
# Visualize sample data
visualize_data_samples(
    train_dataset,
    output_path=f"{CONFIG['output_dir']}/data_samples.png",
    num_samples=2
)

print("✅ Data visualization complete")

## 4️⃣ Model Definition

In [ ]:
# Create model
model = UNet(
    n_channels=3,
    n_classes=2,
    bilinear=CONFIG['bilinear'],
    base_features=CONFIG['base_features']
).to(device)

# Print model info
print_model_info(model, title=f"✅ Model created: {CONFIG['model']}")

In [ ]:
# Define loss function and optimizer
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(
    model.parameters(),
    lr=CONFIG['learning_rate'],
    weight_decay=CONFIG['weight_decay']
)

print("✅ Loss function and optimizer defined")

## 5️⃣ Training

In [ ]:
# Train model
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    config=CONFIG,
    device=device,
    output_dir=CONFIG['output_dir']
)

# Save training history
save_history(history, f"{CONFIG['output_dir']}/training_history.json")

### 📈 Training Curves

In [ ]:
# Plot training curves
plot_training_curves(
    history=history,
    output_path=f"{CONFIG['output_dir']}/training_curves.png"
)

## 6️⃣ Testing

In [ ]:
# Load best model
checkpoint = torch.load(f"{CONFIG['output_dir']}/best_model.pth")
model.load_state_dict(checkpoint['model_state_dict'])
print("✅ Best model loaded")

In [ ]:
# Test model
test_results, predictions, masks = test_model(
    model=model,
    test_loader=test_loader,
    device=device
)

# Print and save results
print_test_results(test_results)
save_test_results(test_results, f"{CONFIG['output_dir']}/test_results.json")

### 🖼️ Visualizations

In [ ]:
# Visualize predictions
visualize_predictions(
    test_dataset=test_dataset,
    predictions=predictions,
    masks=masks,
    output_path=f"{CONFIG['output_dir']}/visualizations/test_predictions.png",
    num_samples=4,
    seed=42
)

## 📊 Results Summary

### Key Findings
- **Overall Dice Score:** See test results above
- **Disc Segmentation:** Generally performs well
- **Cup Segmentation:** More challenging due to smaller size

### Observations
1. Training converges smoothly without preprocessing
2. Model learns meaningful features from raw images
3. Establishes baseline for comparison with enhanced models

### Next Steps
- Experiment 02: Add CLAHE preprocessing
- Experiment 03: Use ResNet encoder with CLAHE

---

**Note:** All results, visualizations, and model checkpoints are saved in the `./results` directory.